# Practical 04: Out-of-Core Batch Ingestion & Aggregation via Chunking

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to ingest a large CSV file exceeding single-process RAM in batches using the chunksize parameter of pd.read_csv(). Calculate the cumulative record count, running sum of transaction amounts, and overall average transaction value across all chunks to prevent Out-Of-Memory (OOM) crashes.

In [1]:
import os
import shutil
import tempfile
import numpy as np
import pandas as pd

TOTAL_ROWS = 900_000
WRITE_BATCH = 300_000
CHUNK_SIZE = 90_000

rng = np.random.default_rng(99)
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "large_transactions.csv")

written = 0
while written < TOTAL_ROWS:
    m = min(WRITE_BATCH, TOTAL_ROWS - written)
    batch = pd.DataFrame({
        "transaction_id": np.arange(written + 1, written + m + 1),
        "customer_id": rng.integers(1, 55000, m),
        "amount": np.round(rng.gamma(2.2, 65.0, m), 2),
        "city": rng.choice(["Kolkata", "Patna", "Ranchi", "Delhi", "Lucknow"], m),
    })
    batch.to_csv(csv_path, mode="a", header=(written == 0), index=False)
    written += m

print(f"File created: {os.path.getsize(csv_path) / 1024**2:.1f} MB, {TOTAL_ROWS:,} rows")

rows_seen = 0
amount_sum = 0.0

chunks = pd.read_csv(csv_path, usecols=["transaction_id", "amount"], chunksize=CHUNK_SIZE)

print(f"{'Chunk':>5} | {'Rows':>8} | {'Cum. records':>12} | {'Running sum':>16} | {'Running avg':>11}")
print("-" * 66)
for i, chunk in enumerate(chunks, start=1):
    rows_seen += len(chunk)
    amount_sum += chunk["amount"].sum()
    print(f"{i:>5} | {len(chunk):>8,} | {rows_seen:>12,} | {amount_sum:>16,.2f} | "
          f"{amount_sum / rows_seen:>11.4f}")

avg_amt = amount_sum / rows_seen
print(f"Total records             : {rows_seen:,}")
print(f"Total transaction amount  : {amount_sum:,.2f}")
print(f"Average transaction value : {avg_amt:.4f}")

full = pd.read_csv(csv_path, usecols=["amount"])["amount"]
print(f"Full-load sum : {full.sum():,.2f} | mean : {full.mean():.4f}")

shutil.rmtree(tmp_dir)
print("Temporary files removed.")

File created: 22.5 MB, 900,000 rows
Chunk |     Rows | Cum. records |      Running sum | Running avg
------------------------------------------------------------------
    1 |   90,000 |       90,000 |    12,880,780.31 |    143.1198
    2 |   90,000 |      180,000 |    25,748,377.03 |    143.0465
    3 |   90,000 |      270,000 |    38,632,507.33 |    143.0834
    4 |   90,000 |      360,000 |    51,516,873.25 |    143.1024
    5 |   90,000 |      450,000 |    64,362,723.41 |    143.0283
    6 |   90,000 |      540,000 |    77,232,628.22 |    143.0234
    7 |   90,000 |      630,000 |    90,078,184.56 |    142.9812
    8 |   90,000 |      720,000 |   102,959,597.53 |    142.9994
    9 |   90,000 |      810,000 |   115,831,438.09 |    143.0018
   10 |   90,000 |      900,000 |   128,720,080.57 |    143.0223
Total records             : 900,000
Total transaction amount  : 128,720,080.57
Average transaction value : 143.0223
Full-load sum : 128,720,080.57 | mean : 143.0223
Temporary files r